# Cálculo de variables objetivo - TIBAITATA

Este notebook preparará las variables necesarias para el cálculo de evapotranspiración de referencia mediante FAO-56.

## Importaciones

In [52]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Configuración de visualización

Se muestran todas las columnas de los DataFrames y se establece una precisión numérica uniforme para facilitar la inspección de resultados.

In [53]:
pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 3)
pd.set_option("display.width", 120)
plt.style.use("seaborn-v0_8-whitegrid")

## Rutas de datos

Se declaran las rutas y se cargan los datasets sin aplicar limpieza ni preprocesamiento.

In [54]:
DATASET_LIMPIO_PATH = "./Datasets/TibaitataLimpio.pkl"
DATASET_ESTACION_PATH = "./Datasets/tibaitata.csv"

df_tibaitata_limpio = pd.read_pickle(DATASET_LIMPIO_PATH)
df_estacion = pd.read_csv(DATASET_ESTACION_PATH)

print("Dataset limpio:", df_tibaitata_limpio.shape)
print("Estacion:", df_estacion.shape)

Dataset limpio: (14545, 9)
Estacion: (1, 20)


## Constantes físicas para FAO-56

- `SOLAR_CONSTANT`: radiación extraterrestre solar, expresada en MJ m⁻² min⁻¹.
- `STEFAN_BOLTZMANN`: constante de Stefan-Boltzmann para la radiación neta de onda larga, en MJ K⁻⁴ m⁻² día⁻¹.
- `LATENT_HEAT_VAPORIZATION`: energía necesaria para vaporizar un kilogramo de agua, en MJ kg⁻¹.
- `SPECIFIC_HEAT_AIR`: calor específico del aire a presión constante, en MJ kg⁻¹ °C⁻¹.
- `EPSILON`: relación entre las masas molares del vapor de agua y el aire seco, adimensional.
- `KELVIN_OFFSET`: desplazamiento para convertir temperaturas de °C a K.
- `SECONDS_PER_DAY`: número de segundos de un día, útil para conversiones de tiempo.


In [55]:
SOLAR_CONSTANT = 0.0820  # MJ m^-2 min^-1
STEFAN_BOLTZMANN = 4.903e-9  # MJ K^-4 m^-2 day^-1
LATENT_HEAT_VAPORIZATION = 2.45  # MJ kg^-1
SPECIFIC_HEAT_AIR = 1.013e-3  # MJ kg^-1 °C^-1
EPSILON = 0.622  # adimensional
KELVIN_OFFSET = 273.15  # K por °C
SECONDS_PER_DAY = 86_400  # s day^-1

## Organización de variables meteorológicas

Se transforma el dataset limpio de formato largo a formato ancho. Antes del pivotado se revisan las claves duplicadas `Fecha`-`parametro_limpio`; si existen, el proceso se detiene para evitar promediar o eliminar registros automáticamente.

In [56]:
parametros_et0 = {
    "Humedad relativa del aire a 2 metros media diaria": "RH_mean",
    "Radiación solar diaria": "Radiation_Wh_m2",
    "Temperatura máxima diaria": "Tmax",
    "Temperatura mínima diaria": "Tmin",
    "Velocidad 10 minutal del viento media diaria": "Wind_speed",
}

datos_et0 = df_tibaitata_limpio[
    df_tibaitata_limpio["parametro_limpio"].isin(parametros_et0)
].copy()

duplicados_et0 = datos_et0[
    datos_et0.duplicated(["Fecha", "parametro_limpio"], keep=False)
].sort_values(["Fecha", "parametro_limpio"])

if duplicados_et0.empty:
    print("No se encontraron duplicados para la combinación Fecha-parametro_limpio.")
else:
    print("Se encontraron los siguientes registros duplicados:")
    display(duplicados_et0[["Fecha", "parametro_limpio", "Valor"]])
    raise ValueError(
        "No se puede realizar el pivotado: existen duplicados para "
        "la combinación Fecha-parametro_limpio."
    )

fechas_disponibles = df_tibaitata_limpio[["Fecha"]].drop_duplicates()

df_tibaitata_et0 = (
    datos_et0.assign(parametro_limpio=datos_et0["parametro_limpio"].map(parametros_et0))
    .pivot(index="Fecha", columns="parametro_limpio", values="Valor")
    .reindex(columns=parametros_et0.values())
    .rename_axis(None, axis=1)
    .reset_index()
)

df_tibaitata_et0 = (
    fechas_disponibles.merge(df_tibaitata_et0, on="Fecha", how="left")
    .sort_values("Fecha")
    .reset_index(drop=True)
)

print("Primeras filas:")
display(df_tibaitata_et0.head())
print("Dimensiones:", df_tibaitata_et0.shape)
print("Tipos de datos:")
display(df_tibaitata_et0.dtypes)
print("Valores nulos por columna:")
display(df_tibaitata_et0.isna().sum())

No se encontraron duplicados para la combinación Fecha-parametro_limpio.
Primeras filas:


,Fecha,RH_mean,Radiation_Wh_m2,Tmax,Tmin,Wind_speed
0,2005-02-02,89.531,1696.5,19.8,4.6,1.645
1,2005-02-03,83.958,4480.5,20.2,5.6,1.965
2,2005-02-04,89.237,2363.3,18.6,6.8,2.149
3,2005-02-05,80.562,3137.0,20.4,9.0,2.558
4,2005-02-06,89.500,1992.7,18.8,10.8,1.965


Dimensiones: (2909, 6)
Tipos de datos:


Fecha              datetime64[us]
RH_mean                   float64
Radiation_Wh_m2           float64
Tmax                      float64
Tmin                      float64
Wind_speed                float64
dtype: object

Valores nulos por columna:


Fecha              0
RH_mean            0
Radiation_Wh_m2    0
Tmax               0
Tmin               0
Wind_speed         0
dtype: int64

In [57]:
columnas_originales_et0 = df_tibaitata_et0.columns.tolist()
numero_registros_et0 = len(df_tibaitata_et0)

# El cálculo vectorizado conserva NaN cuando falta Tmax o Tmin.
df_tibaitata_et0["T_mean"] = (df_tibaitata_et0["Tmax"] + df_tibaitata_et0["Tmin"]) / 2

assert len(df_tibaitata_et0) == numero_registros_et0
assert df_tibaitata_et0.columns[:len(columnas_originales_et0)].tolist() == columnas_originales_et0
assert df_tibaitata_et0["T_mean"].isna().equals(
    df_tibaitata_et0[["Tmax", "Tmin"]].isna().any(axis=1)
)
assert (
    df_tibaitata_et0.loc[df_tibaitata_et0["T_mean"].notna(), "T_mean"]
    == (
        df_tibaitata_et0.loc[df_tibaitata_et0["T_mean"].notna(), "Tmax"]
        + df_tibaitata_et0.loc[df_tibaitata_et0["T_mean"].notna(), "Tmin"]
    ) / 2
).all()

print("Verificaciones: registros y columnas originales conservados; propagación de NaN confirmada.")
display(df_tibaitata_et0[["Tmax", "Tmin", "T_mean"]].head())
print("Valores nulos en T_mean:", df_tibaitata_et0["T_mean"].isna().sum())

Verificaciones: registros y columnas originales conservados; propagación de NaN confirmada.


,Tmax,Tmin,T_mean
0,19.8,4.6,12.2
1,20.2,5.6,12.9
2,18.6,6.8,12.7
3,20.4,9.0,14.7
4,18.8,10.8,14.8


Valores nulos en T_mean: 0


In [58]:
# Presión de vapor de saturación según FAO-56.
def presion_vapor_saturacion(temperatura):
    return 0.6108 * np.exp((17.27 * temperatura) / (temperatura + 237.3))

columnas_antes_es = df_tibaitata_et0.columns.tolist()
valores_antes_es = df_tibaitata_et0[columnas_antes_es].copy()

# Las operaciones vectorizadas conservan NaN cuando falta Tmax o Tmin.
df_tibaitata_et0["es_tmax"] = presion_vapor_saturacion(df_tibaitata_et0["Tmax"])
df_tibaitata_et0["es_tmin"] = presion_vapor_saturacion(df_tibaitata_et0["Tmin"])
df_tibaitata_et0["es"] = (df_tibaitata_et0["es_tmax"] + df_tibaitata_et0["es_tmin"]) / 2

assert df_tibaitata_et0[columnas_antes_es].equals(valores_antes_es)
assert df_tibaitata_et0["es_tmax"].isna().equals(df_tibaitata_et0["Tmax"].isna())
assert df_tibaitata_et0["es_tmin"].isna().equals(df_tibaitata_et0["Tmin"].isna())
assert np.isfinite(df_tibaitata_et0[["es_tmax", "es_tmin", "es"]].dropna()).all().all()
assert (df_tibaitata_et0[["es_tmax", "es_tmin", "es"]].dropna() > 0).all().all()
assert np.allclose(
    df_tibaitata_et0["es"].dropna(),
    ((df_tibaitata_et0["es_tmax"] + df_tibaitata_et0["es_tmin"]) / 2).dropna(),
)

print("Resultados de presión de vapor de saturación (kPa):")
display(df_tibaitata_et0[["Tmax", "Tmin", "es_tmax", "es_tmin", "es"]].head())
print("Valores nulos en las columnas nuevas:")
display(df_tibaitata_et0[["es_tmax", "es_tmin", "es"]].isna().sum())

Resultados de presión de vapor de saturación (kPa):


,Tmax,Tmin,es_tmax,es_tmin,es
0,19.8,4.6,2.309,0.848,1.579
1,20.2,5.6,2.367,0.910,1.638
2,18.6,6.8,2.143,0.988,1.566
3,20.4,9.0,2.397,1.148,1.772
4,18.8,10.8,2.170,1.295,1.733


Valores nulos en las columnas nuevas:


es_tmax    0
es_tmin    0
es         0
dtype: int64

In [59]:
# Presión de vapor de saturación y pendiente a la temperatura media.
columnas_antes_objetivo = df_tibaitata_et0.columns.tolist()
tmean = df_tibaitata_et0["T_mean"]
denominador = tmean + 237.3

with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
    es_tmean = 0.6108 * np.exp((17.27 * tmean) / denominador)
    delta = np.where(
        denominador.ne(0),
        (4098 * es_tmean) / (denominador ** 2),
        np.nan,
    )

df_tibaitata_et0["es_tmean"] = es_tmean
df_tibaitata_et0["delta"] = delta

assert df_tibaitata_et0.columns.tolist()[: len(columnas_antes_objetivo)] == columnas_antes_objetivo
assert df_tibaitata_et0["es_tmean"].isna().equals(tmean.isna())
assert df_tibaitata_et0["delta"].isna().equals(tmean.isna())
assert np.isfinite(df_tibaitata_et0[["es_tmean", "delta"]].dropna()).all().all()
assert (df_tibaitata_et0[["es_tmean", "delta"]].dropna() > 0).all().all()

print("Resultados a T_mean (es_tmean en kPa; delta en kPa/°C):")
display(df_tibaitata_et0[["T_mean", "es_tmean", "delta"]].head())
print("Valores nulos:")
display(df_tibaitata_et0[["T_mean", "es_tmean", "delta"]].isna().sum())

Resultados a T_mean (es_tmean en kPa; delta en kPa/°C):


,T_mean,es_tmean,delta
0,12.2,1.421,0.094
1,12.9,1.488,0.097
2,12.7,1.469,0.096
3,14.7,1.673,0.108
4,14.8,1.684,0.109


Valores nulos:


T_mean      0
es_tmean    0
delta       0
dtype: int64

## Presión real de vapor

Se calcula `ea` mediante la aproximación FAO-56 basada en la humedad relativa media diaria: `ea = es * RH_mean / 100`. La humedad relativa debe estar expresada como porcentaje.

In [60]:
columnas_antes_ea = df_tibaitata_et0.columns.tolist()

assert {"es", "RH_mean"}.issubset(df_tibaitata_et0.columns)
rh_valida = df_tibaitata_et0["RH_mean"].dropna()
assert rh_valida.between(0, 100).all(), "RH_mean debe estar entre 0 y 100 %."
assert rh_valida.max() > 1, "RH_mean parece estar en escala 0-1, no en porcentaje."

df_tibaitata_et0["ea"] = (
    df_tibaitata_et0["es"] * df_tibaitata_et0["RH_mean"] / 100
)

filas_completas_ea = df_tibaitata_et0[["es", "RH_mean"]].notna().all(axis=1)
assert df_tibaitata_et0["ea"].isna().equals(~filas_completas_ea)
assert df_tibaitata_et0.columns[:len(columnas_antes_ea)].tolist() == columnas_antes_ea
assert np.isfinite(df_tibaitata_et0.loc[filas_completas_ea, "ea"]).all()
assert (
    (df_tibaitata_et0.loc[filas_completas_ea, "ea"] >= 0)
    & (df_tibaitata_et0.loc[filas_completas_ea, "ea"] <= df_tibaitata_et0.loc[filas_completas_ea, "es"])
).all()
assert np.allclose(
    df_tibaitata_et0.loc[filas_completas_ea, "ea"],
    df_tibaitata_et0.loc[filas_completas_ea, "es"]
    * df_tibaitata_et0.loc[filas_completas_ea, "RH_mean"]
    / 100,
)

print("Resultados de presión real de vapor (kPa):")
display(df_tibaitata_et0[["RH_mean", "es", "ea"]].head())
print("Valores nulos:")
display(df_tibaitata_et0[["RH_mean", "es", "ea"]].isna().sum())
print("Verificaciones: RH_mean en porcentaje; NaN propagados; 0 <= ea <= es.")

Resultados de presión real de vapor (kPa):


,RH_mean,es,ea
0,89.531,1.579,1.414
1,83.958,1.638,1.376
2,89.237,1.566,1.397
3,80.562,1.772,1.428
4,89.500,1.733,1.551


Valores nulos:


RH_mean    0
es         0
ea         0
dtype: int64

Verificaciones: RH_mean en porcentaje; NaN propagados; 0 <= ea <= es.


## Presión atmosférica

Se calcula la presión atmosférica media de la estación con la ecuación FAO-56 usando exclusivamente la altitud fija de Tibaitatá (`z = 2543 m`).

In [61]:
assert "altitud" in df_estacion.columns
altitudes_estacion = pd.to_numeric(df_estacion["altitud"], errors="coerce").dropna().unique()
assert len(altitudes_estacion) == 1, "La estación debe tener una única altitud válida."
ALTITUD_TIBAITATA_M = altitudes_estacion[0]

assert "P" not in df_tibaitata_et0.columns, "La columna P ya existe; no se sobrescribirá."
columnas_antes_p = df_tibaitata_et0.columns.tolist()
numero_registros_antes_p = len(df_tibaitata_et0)

presion_atmosferica = 101.3 * ((293 - 0.0065 * ALTITUD_TIBAITATA_M) / 293) ** 5.26
df_tibaitata_et0["P"] = presion_atmosferica

assert len(df_tibaitata_et0) == numero_registros_antes_p
assert df_tibaitata_et0.columns[:len(columnas_antes_p)].tolist() == columnas_antes_p
assert np.isfinite(presion_atmosferica)
assert 0 < presion_atmosferica < 101.3
assert df_tibaitata_et0["P"].notna().all()
assert np.isclose(df_tibaitata_et0["P"], presion_atmosferica).all()
assert df_tibaitata_et0["P"].nunique() == 1

print(f"Presión atmosférica calculada para z = {ALTITUD_TIBAITATA_M} m: {presion_atmosferica:.6f} kPa")
print(f"Filas asignadas correctamente: {df_tibaitata_et0['P'].notna().sum()} de {len(df_tibaitata_et0)}")
display(df_tibaitata_et0[["Fecha", "P"]].head())

Presión atmosférica calculada para z = 2543 m: 74.637733 kPa
Filas asignadas correctamente: 2909 de 2909


,Fecha,P
0,2005-02-02,74.638
1,2005-02-03,74.638
2,2005-02-04,74.638
3,2005-02-05,74.638
4,2005-02-06,74.638


## Constante psicrométrica

Se calcula `gamma` (kPa/°C) con la ecuación FAO-56 `gamma = 0.000665 * P`, donde `P` es la presión atmosférica en kPa. Como `P` es constante para la estación, `gamma` también lo es. Si `P` falta en una fecha, `gamma` queda en NaN y no se imputa ningún valor.

In [62]:
assert "P" in df_tibaitata_et0.columns
assert "gamma" not in df_tibaitata_et0.columns, "La columna gamma ya existe; no se sobrescribirá."

columnas_antes_gamma = df_tibaitata_et0.columns.tolist()
valores_antes_gamma = df_tibaitata_et0[columnas_antes_gamma].copy()
fechas_antes_gamma = df_tibaitata_et0["Fecha"].copy()

# La operación vectorizada propaga NaN cuando falta P.
df_tibaitata_et0["gamma"] = 0.000665 * df_tibaitata_et0["P"]

gamma_valida = df_tibaitata_et0["gamma"].dropna()

assert df_tibaitata_et0[columnas_antes_gamma].equals(valores_antes_gamma)
assert df_tibaitata_et0.columns[:len(columnas_antes_gamma)].tolist() == columnas_antes_gamma
assert df_tibaitata_et0["Fecha"].equals(fechas_antes_gamma)
assert df_tibaitata_et0["gamma"].isna().equals(df_tibaitata_et0["P"].isna())
assert np.isfinite(gamma_valida).all()
assert (gamma_valida > 0).all()
assert gamma_valida.nunique() == 1, "gamma debe ser constante."
assert np.allclose(gamma_valida, 0.000665 * df_tibaitata_et0["P"].dropna())

valor_gamma = gamma_valida.iloc[0]
print(f"gamma = {valor_gamma:.6f} kPa/°C (P = {df_tibaitata_et0['P'].iloc[0]:.6f} kPa)")
print("Valores nulos en gamma:", df_tibaitata_et0["gamma"].isna().sum())
print("Verificaciones: columnas y fechas originales conservadas; gamma positiva y constante; NaN propagado desde P.")
display(df_tibaitata_et0[["Fecha", "P", "gamma"]].head())

gamma = 0.049634 kPa/°C (P = 74.637733 kPa)
Valores nulos en gamma: 0
Verificaciones: columnas y fechas originales conservadas; gamma positiva y constante; NaN propagado desde P.


,Fecha,P,gamma
0,2005-02-02,74.638,0.05
1,2005-02-03,74.638,0.05
2,2005-02-04,74.638,0.05
3,2005-02-05,74.638,0.05
4,2005-02-06,74.638,0.05


## Radiación solar observada

Se convierte la radiación solar diaria acumulada de Wh/m²/día a MJ/m²/día con `Rs = Radiation_Wh_m2 * 0.0036` (1 Wh = 3600 J = 0.0036 MJ). No se aplica ninguna otra conversión. Si `Radiation_Wh_m2` falta, `Rs` queda en NaN y no se imputa ningún valor.

In [63]:
assert "Radiation_Wh_m2" in df_tibaitata_et0.columns
assert "Rs" not in df_tibaitata_et0.columns, "La columna Rs ya existe; no se sobrescribirá."

columnas_antes_rs = df_tibaitata_et0.columns.tolist()
valores_antes_rs = df_tibaitata_et0[columnas_antes_rs].copy()
fechas_antes_rs = df_tibaitata_et0["Fecha"].copy()
numero_registros_antes_rs = len(df_tibaitata_et0)

WH_A_MJ = 0.0036  # MJ por Wh

# La operación vectorizada propaga NaN cuando falta la radiación original.
df_tibaitata_et0["Rs"] = df_tibaitata_et0["Radiation_Wh_m2"] * WH_A_MJ

rs_valida = df_tibaitata_et0["Rs"].dropna()

assert len(df_tibaitata_et0) == numero_registros_antes_rs
assert df_tibaitata_et0[columnas_antes_rs].equals(valores_antes_rs)
assert df_tibaitata_et0.columns[:len(columnas_antes_rs)].tolist() == columnas_antes_rs
assert df_tibaitata_et0["Fecha"].equals(fechas_antes_rs)
assert df_tibaitata_et0["Rs"].isna().equals(df_tibaitata_et0["Radiation_Wh_m2"].isna())
assert np.isfinite(rs_valida).all(), "Rs contiene valores infinitos."
assert (rs_valida >= 0).all(), "Rs contiene valores negativos."
assert np.allclose(rs_valida, df_tibaitata_et0["Radiation_Wh_m2"].dropna() * 0.0036)

# Ejemplos numéricos de referencia (1 Wh = 0.0036 MJ).
ejemplos_rs = pd.Series([0.0, 1000.0, 5000.0, 1696.5], name="Radiation_Wh_m2")
esperados_rs = pd.Series([0.0, 3.6, 18.0, 6.1074])
assert np.allclose(ejemplos_rs * WH_A_MJ, esperados_rs)

print("Ejemplos de conversión (Wh/m²/día -> MJ/m²/día):")
display(pd.DataFrame({"Radiation_Wh_m2": ejemplos_rs, "Rs_esperado": esperados_rs}))
print("Primeras filas del dataset:")
display(df_tibaitata_et0[["Fecha", "Radiation_Wh_m2", "Rs"]].head())
print("Valores nulos:")
display(df_tibaitata_et0[["Radiation_Wh_m2", "Rs"]].isna().sum())
print("Resumen de Rs (MJ/m²/día):")
display(rs_valida.describe())
print("Verificaciones: columnas y fechas originales conservadas; NaN propagados; sin infinitos ni negativos.")

Ejemplos de conversión (Wh/m²/día -> MJ/m²/día):


,Radiation_Wh_m2,Rs_esperado
0,0.0,0.000
1,1000.0,3.600
2,5000.0,18.000
3,1696.5,6.107


Primeras filas del dataset:


,Fecha,Radiation_Wh_m2,Rs
0,2005-02-02,1696.5,6.107
1,2005-02-03,4480.5,16.130
2,2005-02-04,2363.3,8.508
3,2005-02-05,3137.0,11.293
4,2005-02-06,1992.7,7.174


Valores nulos:


Radiation_Wh_m2    0
Rs                 0
dtype: int64

Resumen de Rs (MJ/m²/día):


count    2909.000
mean       14.962
std         4.469
min         0.839
25%        11.664
50%        14.596
75%        18.119
max        26.862
Name: Rs, dtype: float64

Verificaciones: columnas y fechas originales conservadas; NaN propagados; sin infinitos ni negativos.


## Variables astronómicas

Se calculan con las ecuaciones FAO-56 las variables que dependen de la fecha y de la latitud de Tibaitatá. La latitud se lee de `df_estacion["latitud"]` (grados decimales, norte positivo):

- `J`: día del año (`dayofyear` de `Fecha`).
- `phi_rad`: latitud en radianes, `phi = latitud * pi / 180`.
- `dr`: distancia relativa inversa Tierra-Sol, `dr = 1 + 0.033 cos(2 pi J / 365)`.
- `solar_declination`: declinación solar en radianes, `delta = 0.409 sin(2 pi J / 365 - 1.39)`.
- `omega_s`: ángulo horario al atardecer en radianes, `omega_s = arccos(-tan(phi) tan(delta))`. El argumento del arcocoseno se limita a [-1, 1] para evitar errores numéricos.

**Tratamiento de años bisiestos.** FAO-56 usa 365 días en las ecuaciones de `dr` y `delta`. Aquí `J` es el `dayofyear` calendario sin ajustes: en años bisiestos, las fechas posteriores al 29 de febrero tienen `J` una unidad mayor que en un año común, y el 31 de diciembre toma `J = 366`. No se elimina ni se reasigna ninguna fecha. El desfase equivale a un día de 365 (aprox. 0.017 rad en el argumento angular), por lo que el efecto sobre `dr`, `delta` y `omega_s` es mínimo. Como la estación está cerca del ecuador, el argumento del arcocoseno queda muy por debajo de 1 en valor absoluto y el límite no debería activarse.

In [64]:
assert "Fecha" in df_tibaitata_et0.columns
columnas_astronomicas = ["J", "phi_rad", "dr", "solar_declination", "omega_s"]
assert not set(columnas_astronomicas) & set(df_tibaitata_et0.columns), "Alguna columna astronómica ya existe; no se sobrescribirá."

columnas_antes_astro = df_tibaitata_et0.columns.tolist()
valores_antes_astro = df_tibaitata_et0[columnas_antes_astro].copy()
numero_registros_antes_astro = len(df_tibaitata_et0)
forma_limpio_antes_astro = df_tibaitata_limpio.shape  # df_tibaitata_limpio no se modifica

# La latitud se toma de df_estacion (columna "latitud", grados decimales; norte positivo).
assert "latitud" in df_estacion.columns
latitudes_estacion = pd.to_numeric(df_estacion["latitud"], errors="coerce").dropna().unique()
assert len(latitudes_estacion) == 1, "La estación debe tener una única latitud válida."
LATITUD_TIBAITATA_DEG = float(latitudes_estacion[0])
assert -90 < LATITUD_TIBAITATA_DEG < 90, "La latitud debe estar en grados entre -90 y 90."

fechas_astro = pd.to_datetime(df_tibaitata_et0["Fecha"])
assert fechas_astro.notna().all(), "Existen fechas faltantes en Fecha."

# Día del año calendario (1-366); en años bisiestos el 31-dic toma J = 366.
J = fechas_astro.dt.dayofyear
phi_rad = np.deg2rad(LATITUD_TIBAITATA_DEG)
angulo_anual = 2 * np.pi * J / 365  # convención FAO-56: 365 días

dr = 1 + 0.033 * np.cos(angulo_anual)
solar_declination = 0.409 * np.sin(angulo_anual - 1.39)

argumento_arccos = -np.tan(phi_rad) * np.tan(solar_declination)
n_argumento_fuera_rango = int(((argumento_arccos < -1) | (argumento_arccos > 1)).sum())
omega_s = np.arccos(np.clip(argumento_arccos, -1, 1))

df_tibaitata_et0["J"] = J
df_tibaitata_et0["phi_rad"] = phi_rad
df_tibaitata_et0["dr"] = dr
df_tibaitata_et0["solar_declination"] = solar_declination
df_tibaitata_et0["omega_s"] = omega_s

# Verificaciones: datos originales intactos.
assert len(df_tibaitata_et0) == numero_registros_antes_astro
assert df_tibaitata_et0[columnas_antes_astro].equals(valores_antes_astro)
assert df_tibaitata_et0.columns[:len(columnas_antes_astro)].tolist() == columnas_antes_astro
assert df_tibaitata_limpio.shape == forma_limpio_antes_astro

# Verificaciones: validez numérica y unidades (radianes).
assert df_tibaitata_et0[columnas_astronomicas].notna().all().all()
assert np.isfinite(df_tibaitata_et0[columnas_astronomicas]).all().all()
assert df_tibaitata_et0["J"].between(1, 366).all()
assert np.isclose(df_tibaitata_et0["phi_rad"], LATITUD_TIBAITATA_DEG * np.pi / 180).all()
assert abs(phi_rad) < np.pi / 2
assert df_tibaitata_et0["dr"].between(1 - 0.033, 1 + 0.033).all()
assert df_tibaitata_et0["solar_declination"].abs().max() <= 0.409 + 1e-12  # rad (~23.45°)
assert df_tibaitata_et0["omega_s"].between(0, np.pi).all()
assert df_tibaitata_et0["omega_s"].max() < 2  # rad; en grados sería > 2 con facilidad

es_bisiesto = fechas_astro.dt.is_leap_year
print("Registros:", len(df_tibaitata_et0))
print("Registros en años bisiestos:", int(es_bisiesto.sum()), "| con J = 366:", int((J == 366).sum()))
print("Argumentos de arccos fuera de [-1, 1] antes de limitar:", n_argumento_fuera_rango)
print(f"phi_rad = {phi_rad:.6f} rad ({LATITUD_TIBAITATA_DEG}°)")
print("Verificaciones: datos originales intactos; valores finitos; ángulos en radianes.")
display(df_tibaitata_et0[["Fecha"] + columnas_astronomicas].head())
display(df_tibaitata_et0[columnas_astronomicas].describe().loc[["min", "mean", "max"]])
print("Valores nulos:")
display(df_tibaitata_et0[columnas_astronomicas].isna().sum())

Registros: 2909
Registros en años bisiestos: 637 | con J = 366: 1
Argumentos de arccos fuera de [-1, 1] antes de limitar: 0
phi_rad = 0.081833 rad (4.688662°)
Verificaciones: datos originales intactos; valores finitos; ángulos en radianes.


,Fecha,J,phi_rad,dr,solar_declination,omega_s
0,2005-02-02,33,0.082,1.028,-0.300,1.545
1,2005-02-03,34,0.082,1.028,-0.295,1.546
2,2005-02-04,35,0.082,1.027,-0.290,1.546
3,2005-02-05,36,0.082,1.027,-0.285,1.547
4,2005-02-06,37,0.082,1.027,-0.280,1.547


,J,phi_rad,dr,solar_declination,omega_s
min,1.000,0.082,0.967,-0.409,1.535
mean,181.858,0.082,1.001,-0.008,1.570
max,366.000,0.082,1.033,0.409,1.606


Valores nulos:


J                    0
phi_rad              0
dr                   0
solar_declination    0
omega_s              0
dtype: int64

## Radiación extraterrestre

Se calcula `Ra` (MJ/m²/día) con la ecuación FAO-56 usando las variables astronómicas ya calculadas (`dr`, `omega_s`, `phi_rad`, `solar_declination`) y la constante `SOLAR_CONSTANT` (`Gsc = 0.0820` MJ/m²/min):

`Ra = (24 * 60 / pi) * Gsc * dr * [omega_s * sin(phi) * sin(delta) + cos(phi) * cos(delta) * sin(omega_s)]`

Todos los ángulos están en radianes. Si falta alguna variable de entrada, `Ra` queda en NaN y no se imputa ningún valor.

In [65]:
columnas_ra_requeridas = ["dr", "omega_s", "phi_rad", "solar_declination"]
assert set(columnas_ra_requeridas).issubset(df_tibaitata_et0.columns)
assert "Ra" not in df_tibaitata_et0.columns, "La columna Ra ya existe; no se sobrescribirá."
assert SOLAR_CONSTANT == 0.0820

columnas_antes_ra = df_tibaitata_et0.columns.tolist()
valores_antes_ra = df_tibaitata_et0[columnas_antes_ra].copy()
numero_registros_antes_ra = len(df_tibaitata_et0)
forma_limpio_antes_ra = df_tibaitata_limpio.shape

dr_ra = df_tibaitata_et0["dr"]
omega_s_ra = df_tibaitata_et0["omega_s"]
phi_ra = df_tibaitata_et0["phi_rad"]
delta_ra = df_tibaitata_et0["solar_declination"]

# Las operaciones vectorizadas propagan NaN si falta alguna variable de entrada.
termino_angular = (
    omega_s_ra * np.sin(phi_ra) * np.sin(delta_ra)
    + np.cos(phi_ra) * np.cos(delta_ra) * np.sin(omega_s_ra)
)
df_tibaitata_et0["Ra"] = (24 * 60 / np.pi) * SOLAR_CONSTANT * dr_ra * termino_angular

ra_valida = df_tibaitata_et0["Ra"].dropna()
filas_completas_ra = df_tibaitata_et0[columnas_ra_requeridas].notna().all(axis=1)

# Datos originales intactos.
assert len(df_tibaitata_et0) == numero_registros_antes_ra
assert df_tibaitata_et0[columnas_antes_ra].equals(valores_antes_ra)
assert df_tibaitata_et0.columns[:len(columnas_antes_ra)].tolist() == columnas_antes_ra
assert df_tibaitata_limpio.shape == forma_limpio_antes_ra

# NaN propagados, valores finitos y positivos.
assert df_tibaitata_et0["Ra"].isna().equals(~filas_completas_ra)
assert np.isfinite(ra_valida).all(), "Ra contiene valores infinitos."
assert (ra_valida > 0).all(), "Ra contiene valores no positivos."

# Rango físico: en latitud ~4.7° N, Ra anual ronda 30-38 MJ/m²/día.
assert ra_valida.between(25, 42).all(), "Ra fuera del rango físico esperado para esta latitud."

# Ra debe ser mayor que Rs observada (la atmósfera solo atenúa la radiación).
rs_comparable = df_tibaitata_et0.loc[filas_completas_ra, ["Rs", "Ra"]].dropna()
n_rs_mayor_ra = int((rs_comparable["Rs"] > rs_comparable["Ra"]).sum())

print("Ra calculada (MJ/m²/día).")
print("Primeras filas:")
display(df_tibaitata_et0[["Fecha", "J", "dr", "omega_s", "solar_declination", "Ra"]].head())
print("Valores nulos:")
display(df_tibaitata_et0[columnas_ra_requeridas + ["Ra"]].isna().sum())
print("Resumen de Ra (MJ/m²/día):")
display(ra_valida.describe())
print("Ra media por mes (MJ/m²/día):")
display(df_tibaitata_et0.groupby(df_tibaitata_et0["Fecha"].dt.month)["Ra"].mean().round(2).to_frame("Ra_media"))
print(f"Registros con Rs > Ra: {n_rs_mayor_ra} de {len(rs_comparable)} (informativo; no se modifica nada).")
print("Verificaciones: columnas y registros originales conservados; NaN propagados; Ra positiva y en rango físico.")

Ra calculada (MJ/m²/día).
Primeras filas:


,Fecha,J,dr,omega_s,solar_declination,Ra
0,2005-02-02,33,1.028,1.545,-0.300,35.335
1,2005-02-03,34,1.028,1.546,-0.295,35.402
2,2005-02-04,35,1.027,1.546,-0.290,35.468
3,2005-02-05,36,1.027,1.547,-0.285,35.535
4,2005-02-06,37,1.027,1.547,-0.280,35.602


Valores nulos:


dr                   0
omega_s              0
phi_rad              0
solar_declination    0
Ra                   0
dtype: int64

Resumen de Ra (MJ/m²/día):


count    2909.000
mean       35.923
std         1.258
min        33.524
25%        35.139
50%        36.067
75%        37.057
max        37.727
Name: Ra, dtype: float64

Ra media por mes (MJ/m²/día):


,Ra_media
Fecha,
1,34.38
2,36.15
3,37.50
4,37.38
5,36.17
6,35.23
7,35.54
8,36.68
9,37.21


Registros con Rs > Ra: 0 de 2909 (informativo; no se modifica nada).
Verificaciones: columnas y registros originales conservados; NaN propagados; Ra positiva y en rango físico.


## Radiación de cielo despejado

Se calcula `Rso` (MJ/m²/día) con la ecuación FAO-56 `Rso = (0.75 + 2e-5 * z) * Ra`, donde `z` es la altitud de la estación (2543 m) y `Ra` la radiación extraterrestre. Si `Ra` falta, `Rso` queda en NaN y no se imputa ningún valor.

In [66]:
assert "Ra" in df_tibaitata_et0.columns
assert "Rso" not in df_tibaitata_et0.columns, "La columna Rso ya existe; no se sobrescribirá."

ALTITUD_RSO_M = 2543  # m
assert np.isclose(ALTITUD_TIBAITATA_M, ALTITUD_RSO_M), "La altitud de la estación no coincide con 2543 m."

columnas_antes_rso = df_tibaitata_et0.columns.tolist()
valores_antes_rso = df_tibaitata_et0[columnas_antes_rso].copy()
numero_registros_antes_rso = len(df_tibaitata_et0)
forma_limpio_antes_rso = df_tibaitata_limpio.shape

# Coeficiente de altitud: 0.75 + 2e-5 * 2543 = 0.80086.
coeficiente_rso = 0.75 + 2e-5 * ALTITUD_RSO_M
assert np.isclose(coeficiente_rso, 0.80086)
assert 0.75 < coeficiente_rso < 1.0

# La operación vectorizada propaga NaN cuando falta Ra.
df_tibaitata_et0["Rso"] = coeficiente_rso * df_tibaitata_et0["Ra"]

rso_valida = df_tibaitata_et0["Rso"].dropna()
ra_con_rso = df_tibaitata_et0.loc[df_tibaitata_et0["Rso"].notna(), "Ra"]

# Datos originales intactos.
assert len(df_tibaitata_et0) == numero_registros_antes_rso
assert df_tibaitata_et0[columnas_antes_rso].equals(valores_antes_rso)
assert df_tibaitata_et0.columns[:len(columnas_antes_rso)].tolist() == columnas_antes_rso
assert df_tibaitata_limpio.shape == forma_limpio_antes_rso

# NaN propagados, valores finitos y positivos.
assert df_tibaitata_et0["Rso"].isna().equals(df_tibaitata_et0["Ra"].isna())
assert np.isfinite(rso_valida).all(), "Rso contiene valores infinitos."
assert (rso_valida > 0).all(), "Rso contiene valores no positivos."

# Coherencia con Ra: Rso < Ra y Rso / Ra constante e igual al coeficiente.
assert (rso_valida < ra_con_rso).all(), "Rso debe ser menor que Ra."
assert np.allclose(rso_valida / ra_con_rso, coeficiente_rso)
assert np.allclose(rso_valida, (0.75 + 2e-5 * 2543) * ra_con_rso)

# Registros con Rs > Rso (informativo; la relación Rs/Rso se usa más adelante).
comparable_rso = df_tibaitata_et0[["Rs", "Rso"]].dropna()
n_rs_mayor_rso = int((comparable_rso["Rs"] > comparable_rso["Rso"]).sum())

print(f"Altitud: {ALTITUD_RSO_M} m | coeficiente (0.75 + 2e-5 * z) = {coeficiente_rso:.5f}")
print("Primeras filas:")
display(df_tibaitata_et0[["Fecha", "Ra", "Rso"]].head())
print("Valores nulos:")
display(df_tibaitata_et0[["Ra", "Rso"]].isna().sum())
print("Resumen de Rso (MJ/m²/día):")
display(rso_valida.describe())
print(f"Razón Rso/Ra: mín = {(rso_valida / ra_con_rso).min():.5f}, máx = {(rso_valida / ra_con_rso).max():.5f}")
print(f"Registros con Rs > Rso: {n_rs_mayor_rso} de {len(comparable_rso)} (informativo; no se modifica nada).")
print("Verificaciones: columnas y registros originales conservados; NaN propagados; Rso positiva y menor que Ra.")

Altitud: 2543 m | coeficiente (0.75 + 2e-5 * z) = 0.80086
Primeras filas:


,Fecha,Ra,Rso
0,2005-02-02,35.335,28.299
1,2005-02-03,35.402,28.352
2,2005-02-04,35.468,28.405
3,2005-02-05,35.535,28.459
4,2005-02-06,35.602,28.512


Valores nulos:


Ra     0
Rso    0
dtype: int64

Resumen de Rso (MJ/m²/día):


count    2909.000
mean       28.770
std         1.007
min        26.848
25%        28.141
50%        28.884
75%        29.678
max        30.214
Name: Rso, dtype: float64

Razón Rso/Ra: mín = 0.80086, máx = 0.80086
Registros con Rs > Rso: 0 de 2909 (informativo; no se modifica nada).
Verificaciones: columnas y registros originales conservados; NaN propagados; Rso positiva y menor que Ra.
